# Assignment 1 — Gradio dashboard

## 1. Setup

In [1]:
import json
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn

ARTIFACTS_DIR = Path("artifacts")

/Users/joseluishervasdecela_1/daiist-assignment-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load what training saved

First the settings and tables: feature names, the cost of each error, the thresholds chosen on validation, the test results and the feature-group table.

In [2]:
with open(ARTIFACTS_DIR / "metadata.json") as f:
    meta = json.load(f)

FEATURE_NAMES = meta["feature_names"]
COST_FN = meta["cost_fn"]
COST_FP = meta["cost_fp"]
CHOSEN_THRESHOLD = meta["chosen_threshold"]

data = pd.read_csv(ARTIFACTS_DIR / "dashboard_data.csv", parse_dates=["created"])
test_results = pd.read_csv(ARTIFACTS_DIR / "test_results.csv")
ablation = pd.read_csv(ARTIFACTS_DIR / "ablation.csv")
print(data.shape, "| features:", len(FEATURE_NAMES))

(3395, 23) | features: 15


## 3. Load the three models

In [3]:
scaler = joblib.load(ARTIFACTS_DIR / "scaler.joblib")
sk_model = joblib.load(ARTIFACTS_DIR / "sklearn_model.joblib")

manual_params = torch.load(ARTIFACTS_DIR / "manual_torch_model.pt")
w_manual, b_manual = manual_params["w"], manual_params["b"]

std_model = nn.Sequential(nn.Linear(len(FEATURE_NAMES), 1), nn.Sigmoid())
std_model.load_state_dict(torch.load(ARTIFACTS_DIR / "standard_torch_model.pt"))
std_model.eval()

Sequential(
  (0): Linear(in_features=15, out_features=1, bias=True)
  (1): Sigmoid()
)

## 4. Predictions for every session

I scale the features with the saved scaler (the same one fitted on train) and get the probability of a long session from each model. This is inference only.

In [5]:
X = scaler.transform(data[FEATURE_NAMES].to_numpy(dtype=float))
X_t = torch.tensor(X, dtype=torch.float32)

with torch.no_grad():
    data["p_sklearn"] = sk_model.predict_proba(X)[:, 1]
    data["p_manual_torch"] = torch.sigmoid(X_t @ w_manual + b_manual).numpy()
    data["p_standard_torch"] = std_model(X_t).squeeze(-1).numpy()

MODELS = ["sklearn", "manual_torch", "standard_torch"]
SPLIT_LABELS = {"Test": "test", "Validation": "val", "Train": "train"}
# Fixed order and colours so "short" and "long" look the same in every chart
OUTCOME_ORDER = ["short (≤ 4 h)", "long (> 4 h)"]
OUTCOME_COLORS = {"short (≤ 4 h)": "#4C78A8", "long (> 4 h)": "#E45756"}
print(data[["target"] + [f"p_{m}" for m in MODELS]].describe().round(3))


         target  p_sklearn  p_manual_torch  p_standard_torch
count  3395.000   3395.000        3395.000          3395.000
mean      0.112      0.115           0.116             0.116
std       0.315      0.110           0.110             0.110
min       0.000      0.001           0.001             0.001
25%       0.000      0.050           0.050             0.050
50%       0.000      0.088           0.088             0.088
75%       0.000      0.136           0.136             0.136
max       1.000      0.942           0.942             0.942


## 5. Helper functions

In [6]:
def confusion_counts(y_true, probs, threshold):
    alert = probs >= threshold
    tp = int((alert & (y_true == 1)).sum())
    fp = int((alert & (y_true == 0)).sum())
    fn = int((~alert & (y_true == 1)).sum())
    tn = int((~alert & (y_true == 0)).sum())
    return tp, fp, fn, tn

def business_cost(y_true, probs, threshold, cost_fn=COST_FN, cost_fp=COST_FP):
    tp, fp, fn, tn = confusion_counts(y_true, probs, threshold)
    return cost_fn * fn + cost_fp * fp

## 6. Tab Compare models: prediction vs. actual

For a classifier, "prediction vs. actual" means: when a model says 30%, do about 30% of those sessions really turn out long? I sort the sessions into 10 groups by predicted probability and plot the average prediction against the real share of long sessions. If the three models are really the same, their lines should sit on top of each other. The second plot shows the predicted probabilities split by what actually happened: long sessions should get higher probabilities.

In [7]:
def calibration_plot(split_label):
    d = data[data["split"] == SPLIT_LABELS[split_label]]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[0, 0.7], y=[0, 0.7], mode="lines", name="perfect",
                             line=dict(dash="dot", color="grey")))
    for m, dash in zip(MODELS, ["solid", "dash", "dot"]):
        bins = pd.qcut(d[f"p_{m}"], q=10, duplicates="drop")
        g = d.groupby(bins, observed=True).agg(pred=(f"p_{m}", "mean"), actual=("target", "mean"))
        fig.add_trace(go.Scatter(x=g["pred"], y=g["actual"], mode="lines+markers",
                                 name=m, line=dict(dash=dash)))
    fig.update_layout(title=f"Predicted vs. actual share of long sessions ({split_label})",
                      xaxis_title="average predicted probability",
                      yaxis_title="actual share of sessions > 4 h", height=450)
    return fig

def probability_boxplot(split_label):
    d = data[data["split"] == SPLIT_LABELS[split_label]]
    long_df = d.melt(id_vars="target", value_vars=[f"p_{m}" for m in MODELS],
                     var_name="model", value_name="predicted probability")
    long_df["model"] = long_df["model"].str.replace("p_", "", regex=False)
    long_df["what happened"] = long_df["target"].map({0: "short (≤ 4 h)", 1: "long (> 4 h)"})
    fig = px.box(long_df, x="model", y="predicted probability", color="what happened",
                 category_orders={"what happened": OUTCOME_ORDER}, color_discrete_map=OUTCOME_COLORS,
                 title=f"Predicted probability by actual outcome ({split_label})")
    fig.update_layout(height=450)
    return fig

def max_difference_text(split_label):
    d = data[data["split"] == SPLIT_LABELS[split_label]]
    diff_manual = np.abs(d["p_manual_torch"] - d["p_sklearn"]).max()
    diff_std = np.abs(d["p_standard_torch"] - d["p_sklearn"]).max()
    return (f"**Largest difference in predicted probability vs. scikit-learn ({split_label}):** "
            f"manual PyTorch {diff_manual:.4f}, standard PyTorch {diff_std:.4f}.")

## 7. Tab Data: feature and target distributions

Pick a column and a split to see how it's distributed, coloured by whether the session ended up long or short. Percentages are within each colour, so long and short sessions can be compared even though long ones are rare. There's also a small chart with the share of long sessions in each split.

In [ ]:
# Fixed order and colours so "short" and "long" look the same in every chart
OUTCOME_ORDER = ["short (≤ 4 h)", "long (> 4 h)"]
OUTCOME_COLORS = {"short (≤ 4 h)": "#4C78A8", "long (> 4 h)": "#E45756"}

DISTRIBUTION_COLUMNS = ["chargeTimeHrs", "startTime", "distance_raw", "weekday", "facilityType",
                        "user_prev_long_share", "user_prev_mean_hours", "user_prev_sessions",
                        "distance_missing", "manager_vehicle", "reported_zip"]

def distribution_plot(column, split_label):
    d = data if split_label == "All" else data[data["split"] == SPLIT_LABELS[split_label]]
    d = d.assign(**{"what happened": d["target"].map({0: "short (≤ 4 h)", 1: "long (> 4 h)"})})
    fig = px.histogram(d, x=column, color="what happened", barmode="overlay",
                       histnorm="percent", opacity=0.65,
                       category_orders={"what happened": OUTCOME_ORDER},
                       color_discrete_map=OUTCOME_COLORS,
                       title=f"{column}: distribution by outcome ({split_label})")
    if column == "chargeTimeHrs":
        fig.add_vline(x=meta["long_session_hours"], line_dash="dash", line_color="black")
        fig.update_xaxes(range=[0, 12])   # a few sessions last 50+ hours and squash the rest
    fig.update_layout(height=450, yaxis_title="% of sessions in each group")
    return fig

def target_by_split_plot():
    g = data.groupby("split", sort=False)["target"].mean().reindex(["train", "val", "test"])
    fig = px.bar(x=g.index, y=g.values, text=[f"{v:.1%}" for v in g.values],
                 labels={"x": "split", "y": "share of sessions > 4 h"},
                 title="Target: share of long sessions in each split")
    fig.update_layout(height=350)
    return fig

## 8. Tab Threshold & cost

This is the business part. Move the threshold and see what happens: the confusion matrix, how many reminders get sent, how many blocking sessions slip through, and the total cost. You can also change how much each error costs, in case the company sees it differently than I do. The last chart shows the cost for every threshold with a marker where the slider is.

In [9]:
def threshold_view(model, split_label, threshold, cost_fn, cost_fp):
    cost_fn, cost_fp = cost_fn or 0, cost_fp or 0   # an empty box counts as 0
    d = data[data["split"] == SPLIT_LABELS[split_label]]
    y, p = d["target"].to_numpy(), d[f"p_{model}"].to_numpy()
    tp, fp, fn, tn = confusion_counts(y, p, threshold)
    cost = business_cost(y, p, threshold, cost_fn, cost_fp)
    cost_nobody = cost_fn * int(y.sum())
    recall = tp / (tp + fn) if tp + fn else 0.0
    precision = tp / (tp + fp) if tp + fp else 0.0

    cm = go.Figure(go.Heatmap(
        z=[[tn, fp], [fn, tp]],
        x=["no reminder", "reminder"], y=["short (≤ 4 h)", "long (> 4 h)"],
        text=[[f"{tn}<br>correctly ignored", f"{fp}<br>unnecessary reminders"],
              [f"{fn}<br>missed (colleague can't charge)", f"{tp}<br>caught"]],
        texttemplate="%{text}", colorscale="Blues", showscale=False))
    cm.update_layout(title=f"Confusion matrix — {model}, {split_label}, threshold {threshold:.2f}",
                     xaxis_title="model decision", yaxis_title="what actually happened",
                     yaxis_autorange="reversed", height=400)

    summary = (
        f"### Business cost: **{cost}**\n"
        f"{fn} missed long sessions × {cost_fn} + {fp} unnecessary reminders × {cost_fp}\n\n"
        f"Reminding nobody would cost **{cost_nobody}**, so the model saves "
        f"**{cost_nobody - cost}** ({(cost_nobody - cost) / max(cost_nobody, 1):.0%}).\n\n"
        f"Recall: **{recall:.0%}** of long sessions caught · Precision: **{precision:.0%}** "
        f"of reminders were justified · Reminders sent: **{tp + fp}** of {len(y)} sessions"
    )

    grid = np.round(np.arange(0.01, 0.91, 0.01), 2)
    costs = [business_cost(y, p, t, cost_fn, cost_fp) for t in grid]
    curve = go.Figure(go.Scatter(x=grid, y=costs, mode="lines", name="cost"))
    curve.add_trace(go.Scatter(x=[threshold], y=[cost], mode="markers", name="current threshold",
                               marker=dict(size=12, color="red")))
    curve.add_vline(x=CHOSEN_THRESHOLD[model], line_dash="dash", line_color="green",
                    annotation_text="chosen on validation")
    curve.update_layout(title="Business cost for every threshold", xaxis_title="threshold",
                        yaxis_title="business cost", height=400)
    return cm, summary, curve

In [10]:
with gr.Blocks(title="EV charger blocking: dashboard") as demo:
    gr.Markdown(
        "# Will this charging session block the charger?\n"
        "Predicting, when a car is plugged in, whether the session will last more than "
        f"**{meta['long_session_hours']} hours**, so the company can send a reminder. "
        "Same logistic regression trained three ways. Models are loaded from `artifacts/`, "
        "nothing is retrained here."
    )

    with gr.Tab("Overview"):
        gr.Markdown(f"### Results on the test set (threshold chosen on validation, "
                    f"cost: missed = {COST_FN}, unnecessary reminder = {COST_FP})")
        gr.Dataframe(test_results.round(3), interactive=False)
        gr.Markdown("### How much each feature group helps (validation)")
        gr.Dataframe(ablation.round(3), interactive=False)

    with gr.Tab("Compare models"):
        split_cmp = gr.Radio(["Test", "Validation", "Train"], value="Test", label="Data split")
        diff_md = gr.Markdown()
        with gr.Row():
            calib = gr.Plot()
            box = gr.Plot()
        split_cmp.change(calibration_plot, split_cmp, calib)
        split_cmp.change(probability_boxplot, split_cmp, box)
        split_cmp.change(max_difference_text, split_cmp, diff_md)

    with gr.Tab("Data"):
        with gr.Row():
            col_dd = gr.Dropdown(DISTRIBUTION_COLUMNS, value="chargeTimeHrs", label="Column")
            split_dist = gr.Radio(["All", "Train", "Validation", "Test"], value="All", label="Data split")
        dist = gr.Plot()
        target_plot = gr.Plot()
        col_dd.change(distribution_plot, [col_dd, split_dist], dist)
        split_dist.change(distribution_plot, [col_dd, split_dist], dist)

    with gr.Tab("Threshold & cost"):
        with gr.Row():
            model_dd = gr.Dropdown(MODELS, value="sklearn", label="Model")
            split_thr = gr.Radio(["Test", "Validation"], value="Test", label="Data split")
        thr = gr.Slider(0.01, 0.90, value=CHOSEN_THRESHOLD["sklearn"], step=0.01, label="Decision threshold")
        with gr.Row():
            cfn = gr.Number(value=COST_FN, label="Cost of a missed long session (FN)")
            cfp = gr.Number(value=COST_FP, label="Cost of an unnecessary reminder (FP)")
        summary_md = gr.Markdown()
        with gr.Row():
            cm_plot = gr.Plot()
            curve_plot = gr.Plot()
        inputs = [model_dd, split_thr, thr, cfn, cfp]
        outputs = [cm_plot, summary_md, curve_plot]
        for control in inputs:
            control.change(threshold_view, inputs, outputs)

    def draw_all():
        return (calibration_plot("Test"), probability_boxplot("Test"), max_difference_text("Test"),
                distribution_plot("chargeTimeHrs", "All"), target_by_split_plot(),
                *threshold_view("sklearn", "Test", CHOSEN_THRESHOLD["sklearn"], COST_FN, COST_FP))

    demo.load(draw_all, None, [calib, box, diff_md, dist, target_plot, cm_plot, summary_md, curve_plot])

In [ ]:
demo.launch()